# TS Erin vorticity-budget figure reproduction

This notebook is a pared-down version of the exploratory `vortbudget.ipynb`. It keeps only the code needed to reproduce the manuscript RMW-relative vorticity-budget summary derived from this notebook.

The workflow is:

1. Load or build rainfall/RMW radial caches.
2. Load or build 0–1-km vorticity-budget radial caches.
3. Build the four peak-centered profiles.
4. Plot the inside-RMW versus outside-RMW budget summary.

For GitHub reproduction, include the derived `cache/*.pkl` files and leave `OVERWRITE_CACHE=False` and `REFRESH_RMW_ON_LOAD=False`. Rebuilding the caches from scratch requires the stitched WRF output and track NetCDF files listed in the settings cell.

The outside-RMW statistic is defined here as **RMW to 200 km**, matching the manuscript text.


In [ ]:
import os
import pickle
from pathlib import Path
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import wrf


In [ ]:
# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------
# This notebook can either rebuild the derived caches from the stitched WRF
# output or simply load existing cache/*.pkl files. For GitHub reproduction,
# leave OVERWRITE_CACHE=False and REFRESH_RMW_ON_LOAD=False so the notebook can
# regenerate the figure directly from the included derived cache files.

cache_dir = "cache"
figure_dir = "figures"
os.makedirs(cache_dir, exist_ok=True)
os.makedirs(figure_dir, exist_ok=True)

OVERWRITE_CACHE = False
REFRESH_RMW_ON_LOAD = False

# Stitched WRF files. These are only required if cache files are absent or if
# OVERWRITE_CACHE=True. They are not needed when reproducing from included caches.
directory_c = '/ourdisk/hpc/radclouds/auto_archive_notyet/tape_2copies/tc_erin/stitchd02_wrfout.nc'
directory_d = '/ourdisk/hpc/radclouds/auto_archive_notyet/tape_2copies/tc_erin/stitchd02_wrfoutdiurnal.nc'
directory_dimp = '/ourdisk/hpc/radclouds/auto_archive_notyet/tape_2copies/tc_erin/diurnal_imp/output/stitchd02_wrfoutimp_d.nc'

track_file_control = "/ourdisk/hpc/radclouds/auto_archive_notyet/tape_2copies/tc_erin/analysis/erin3000track_rvor.nc"
track_file_diurnal = "/ourdisk/hpc/radclouds/auto_archive_notyet/tape_2copies/tc_erin/analysis/erin_d3000track_rvor.nc"
track_file_dimp = "/ourdisk/hpc/radclouds/auto_archive_notyet/tape_2copies/tc_erin/analysis/erindimp0409253000track_rvor.nc"

runnamelistuse = ['control', 'diurnal', 'd_imp1']

trackfilesuse = {
    'control': track_file_control,
    'diurnal': track_file_diurnal,
    'd_imp1': track_file_dimp,
}

directoriesuse = {
    'control': directory_c,
    'diurnal': directory_d,
    'd_imp1': directory_dimp,
}

# Plotting and analysis settings
r_edges_km = np.arange(0, 251, 5)
rmw_search_km = (10, 150)
timerange = ("2007-08-18 00:00:00", "2007-08-20 06:00:00")
future_hours = 6
rain_window_hours = 3
z1_budget = 0.0
z2_budget = 1000.0
rmin_stat_km = 15.0
rmax_stat_km = 200.0

# Peak times used in the manuscript RMW-relative budget summary.
peak_specs = {
    "control": {
        "runname": "control",
        "time": "2007-08-19 06:00:00",
        "title": "Control\n2007-08-19 06:00 UTC",
    },
    "diurnal": {
        "runname": "diurnal",
        "time": "2007-08-18 18:00:00",
        "title": "Diurnal\n2007-08-18 18:00 UTC",
    },
    "dimp_first": {
        "runname": "d_imp1",
        "time": "2007-08-18 18:00:00",
        "title": "D_imp1 first peak\n2007-08-18 18:00 UTC",
    },
    "dimp_second": {
        "runname": "d_imp1",
        "time": "2007-08-19 06:00:00",
        "title": "D_imp1 second peak\n2007-08-19 06:00 UTC",
    },
}


## Helper functions

In [ ]:

def save_pkl(obj, filename):
    with open(filename, "wb") as f:
        pickle.dump(obj, f)


def load_pkl(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def _get_times(ds):
    times_raw = wrf.getvar(ds, "times", timeidx=wrf.ALL_TIMES)
    return pd.to_datetime(np.array(times_raw).astype(str), errors="coerce")


def _read_track(track_file):
    with Dataset(track_file) as nc:
        clat = np.asarray(nc.variables["clat"][:], dtype=float)
        clon = np.asarray(nc.variables["clon"][:], dtype=float)
    return clat, clon


def _latlon_to_polar(lat2d, lon2d, clat, clon):
    dlat = lat2d - clat
    dlon = (lon2d - clon + 180.0) % 360.0 - 180.0

    dy_km = 111.0 * dlat
    dx_km = 111.0 * np.cos(np.deg2rad(clat)) * dlon

    radius_km = np.sqrt(dx_km**2 + dy_km**2)
    theta = np.arctan2(dy_km, dx_km)
    radius_deg = np.sqrt(dlon**2 + dlat**2)
    return radius_km, theta, radius_deg


def _radial_mean(field2d, radius_km, r_edges_km):
    out = np.full(len(r_edges_km) - 1, np.nan)
    for i in range(len(r_edges_km) - 1):
        mask = (
            (radius_km >= r_edges_km[i]) &
            (radius_km < r_edges_km[i + 1]) &
            np.isfinite(field2d)
        )
        if np.any(mask):
            out[i] = np.nanmean(field2d[mask])
    return out


def _area_weighted_integral(profile, r_centers, rmin=0.0, rmax=None):
    if rmax is None:
        rmax = np.nanmax(r_centers)

    mask = (
        np.isfinite(profile) &
        np.isfinite(r_centers) &
        (r_centers >= rmin) &
        (r_centers <= rmax)
    )
    if np.count_nonzero(mask) < 2:
        return np.nan

    return np.trapz(profile[mask] * r_centers[mask], r_centers[mask])


def _compute_tangential_wind(u, v, theta):
    return -u * np.sin(theta) + v * np.cos(theta)


def _interp_to_r(r_src, y_src, r_target):
    r_src = np.asarray(r_src, dtype=float)
    y_src = np.asarray(y_src, dtype=float)
    r_target = np.asarray(r_target, dtype=float)

    good = np.isfinite(r_src) & np.isfinite(y_src)
    if np.sum(good) < 2:
        return np.full_like(r_target, np.nan, dtype=float)

    out = np.interp(r_target, r_src[good], y_src[good])
    outside = (r_target < np.nanmin(r_src[good])) | (r_target > np.nanmax(r_src[good]))
    out[outside] = np.nan
    return out


def _get_time_indices(times_all, timerange=None):
    if timerange is None:
        return np.arange(len(times_all))

    t0 = pd.to_datetime(timerange[0])
    t1 = pd.to_datetime(timerange[1])
    keep = (times_all >= t0) & (times_all <= t1)
    return np.where(keep)[0]


def _compute_rmw_from_tangential(u10, v10, theta, radius_km, r_edges_km, rmw_search_km):
    vt10 = _compute_tangential_wind(u10, v10, theta)
    vt_prof = _radial_mean(vt10, radius_km, r_edges_km)
    r_centers = 0.5 * (r_edges_km[:-1] + r_edges_km[1:])

    search = (
        np.isfinite(vt_prof) &
        (r_centers >= rmw_search_km[0]) &
        (r_centers <= rmw_search_km[1])
    )
    if not np.any(search):
        return np.nan

    return float(r_centers[search][np.nanargmax(vt_prof[search])])


def _build_rmw_series(runname, directories, trackfiles, r_edges_km, rmw_search_km, times=None):
    ds = Dataset(directories[runname])
    times_all = pd.DatetimeIndex(_get_times(ds))
    lat2d = np.asarray(ds.variables["XLAT"][0], dtype=float)
    lon2d = np.asarray(ds.variables["XLONG"][0], dtype=float)
    clat_all, clon_all = _read_track(trackfiles[runname])

    if times is None:
        time_indices = np.arange(len(times_all))
        out_times = times_all[time_indices]
    else:
        out_times = pd.DatetimeIndex(times)
        time_indices = np.array([int(np.argmin(np.abs(times_all - t))) for t in out_times])

    rmw_km = np.full(len(time_indices), np.nan)
    for n, it in enumerate(time_indices):
        radius_km, theta, _ = _latlon_to_polar(lat2d, lon2d, clat_all[it], clon_all[it])
        u10 = np.asarray(ds.variables["U10"][it], dtype=float)
        v10 = np.asarray(ds.variables["V10"][it], dtype=float)
        rmw_km[n] = _compute_rmw_from_tangential(
            u10=u10,
            v10=v10,
            theta=theta,
            radius_km=radius_km,
            r_edges_km=r_edges_km,
            rmw_search_km=rmw_search_km,
        )

    ds.close()
    return pd.DatetimeIndex(out_times), rmw_km


def _refresh_cached_rmw(cache, runname, directories, trackfiles, cache_file=None):
    if cache.get("rmw_method") == "tangential_wind":
        return cache

    print(f"[{runname}] updating cached RMW to tangential-wind method")
    r_edges_km = np.asarray(cache["r_edges_km"], dtype=float)
    rmw_search_km = tuple(cache.get("rmw_search_km", (10, 150)))
    _, rmw_km = _build_rmw_series(
        runname=runname,
        directories=directories,
        trackfiles=trackfiles,
        r_edges_km=r_edges_km,
        rmw_search_km=rmw_search_km,
        times=pd.DatetimeIndex(cache["times"]),
    )
    cache["rmw_km"] = rmw_km
    cache["rmw_search_km"] = rmw_search_km
    cache["rmw_method"] = "tangential_wind"

    if cache_file is not None:
        save_pkl(cache, cache_file)

    return cache


In [ ]:

def _layer_massweighted_mean(field3d, rho3d, z3d, z1, z2):
    dz = np.abs(np.gradient(z3d, axis=0))
    mask = (
        np.isfinite(field3d) &
        np.isfinite(rho3d) &
        np.isfinite(z3d) &
        np.isfinite(dz) &
        (z3d >= z1) &
        (z3d <= z2)
    )

    num = np.sum(np.where(mask, rho3d * field3d * dz, 0.0), axis=0)
    den = np.sum(np.where(mask, rho3d * dz, 0.0), axis=0)

    out = np.full_like(num, np.nan, dtype=float)
    good = den > 0
    out[good] = num[good] / den[good]
    return out

def _destagger_uv(ds, it):
    u_stag = np.asarray(ds.variables["U"][it], dtype=float)
    v_stag = np.asarray(ds.variables["V"][it], dtype=float)

    u = 0.5 * (u_stag[:, :, :-1] + u_stag[:, :, 1:])
    v = 0.5 * (v_stag[:, :-1, :] + v_stag[:, 1:, :])
    return u, v

def _compute_density_3d(ds, it):
    p_pa = np.asarray(ds.variables["P"][it] + ds.variables["PB"][it], dtype=float)
    qv = np.asarray(ds.variables["QVAPOR"][it], dtype=float)
    tk = np.asarray(wrf.getvar(ds, "tk", timeidx=it), dtype=float)
    tv = tk * (1.0 + 0.61 * qv)
    return p_pa / (287.04 * tv)

def _scale_avo_to_s1(avor):
    avor = np.asarray(avor, dtype=float)
    if np.nanmedian(np.abs(avor)) > 1e-2:
        avor = avor * 1e-5
    return avor

def _storm_motion_ms(times, clat, clon):
    times = pd.DatetimeIndex(times)
    tsec = np.asarray((times - times[0]).total_seconds(), dtype=float)

    lat = np.deg2rad(np.asarray(clat, dtype=float))
    lon = np.unwrap(np.deg2rad(np.asarray(clon, dtype=float)))

    re = 6371000.0
    dlat_dt = np.gradient(lat, tsec, edge_order=2)
    dlon_dt = np.gradient(lon, tsec, edge_order=2)

    cy = re * dlat_dt
    cx = re * np.cos(lat) * dlon_dt
    return cx, cy

def _ddz_3d(var3d, z3d):
    var3d = np.asarray(var3d, dtype=float)
    z3d = np.asarray(z3d, dtype=float)

    out = np.full_like(var3d, np.nan, dtype=float)

    dvar = np.empty_like(var3d, dtype=float)
    dz = np.empty_like(z3d, dtype=float)

    dvar[1:-1] = var3d[2:] - var3d[:-2]
    dz[1:-1] = z3d[2:] - z3d[:-2]

    dvar[0] = var3d[1] - var3d[0]
    dz[0] = z3d[1] - z3d[0]

    dvar[-1] = var3d[-1] - var3d[-2]
    dz[-1] = z3d[-1] - z3d[-2]

    good = np.isfinite(dvar) & np.isfinite(dz) & (np.abs(dz) > 0)
    out[good] = dvar[good] / dz[good]
    return out

def _apply_radius_mask(field2d, radius_deg, rmin_deg=0.0, rmax_deg=1.0):
    mask = (
        np.isfinite(field2d) &
        np.isfinite(radius_deg) &
        (radius_deg >= rmin_deg) &
        (radius_deg < rmax_deg)
    )
    if np.any(mask):
        return np.nanmean(field2d[mask])
    return np.nan

def _time_gradient_1d(y, times):
    times = pd.DatetimeIndex(times)
    tsec = np.asarray((times - times[0]).total_seconds(), dtype=float)
    out = np.full_like(np.asarray(y, dtype=float), np.nan, dtype=float)
    good = np.isfinite(y)
    if np.count_nonzero(good) >= 3:
        out[good] = np.gradient(np.asarray(y, dtype=float)[good], tsec[good], edge_order=2)
    return out

In [ ]:

def _compute_budget_terms_3d(ds, it, cx_ms=0.0, cy_ms=0.0):
    dx = float(getattr(ds, "DX"))
    dy = float(getattr(ds, "DY"))

    u, v = _destagger_uv(ds, it)
    w = np.asarray(wrf.getvar(ds, "wa", timeidx=it), dtype=float)
    z_agl = np.asarray(wrf.getvar(ds, "height_agl", timeidx=it), dtype=float)
    rho = _compute_density_3d(ds, it)

    avor = _scale_avo_to_s1(np.asarray(wrf.getvar(ds, "avo", timeidx=it), dtype=float))

    u_rel = u - cx_ms
    v_rel = v - cy_ms

    davor_dx = np.gradient(avor, dx, axis=2, edge_order=2)
    davor_dy = np.gradient(avor, dy, axis=1, edge_order=2)

    dudx = np.gradient(u, dx, axis=2, edge_order=2)
    dvdy = np.gradient(v, dy, axis=1, edge_order=2)
    dwdx = np.gradient(w, dx, axis=2, edge_order=2)
    dwdy = np.gradient(w, dy, axis=1, edge_order=2)

    delta = dudx + dvdy

    davor_dz = _ddz_3d(avor, z_agl)
    dudz = _ddz_3d(u, z_agl)
    dvdz = _ddz_3d(v, z_agl)

    advh = -(u_rel * davor_dx + v_rel * davor_dy)
    stretch = -(avor * delta)
    vadv = -(w * davor_dz)
    tilt = -(dvdz * dwdx) + (dudz * dwdy)
    resolved = advh + stretch + vadv + tilt

    return {
        "avor": avor,
        "advh": advh,
        "stretch": stretch,
        "vadv": vadv,
        "tilt": tilt,
        "resolved": resolved,
        "rho": rho,
        "z_agl": z_agl,
    }

In [ ]:

def _compute_rain_rate(ds, times_all, it):
    lat2d = np.asarray(ds.variables["XLAT"][0, :, :], dtype=float)
    if it == 0:
        return np.full_like(lat2d, np.nan, dtype=float)

    rain_now = np.asarray(ds.variables["RAINC"][it] + ds.variables["RAINNC"][it], dtype=float)
    rain_prev = np.asarray(ds.variables["RAINC"][it - 1] + ds.variables["RAINNC"][it - 1], dtype=float)
    dt_hr = (times_all[it] - times_all[it - 1]) / pd.Timedelta(hours=1)

    rain_rate = (rain_now - rain_prev) / dt_hr
    rain_rate = np.where(rain_rate < 0, np.nan, rain_rate)
    return rain_rate


def build_rain_rmw_radial_cache(
    runname,
    directories,
    trackfiles,
    cache_dir="cache",
    r_edges_km=np.arange(0, 251, 5),
    timerange=None,
    rmw_search_km=(10, 150),
    rain_radius_max_km=200,
    overwrite=False,
    refresh_rmw_on_load=False,
):
    cache_file = os.path.join(cache_dir, f"{runname}_rain_rmw_radial.pkl")
    if os.path.exists(cache_file) and not overwrite:
        print(f"[{runname}] loading existing rain radial cache")
        cache = load_pkl(cache_file)
        if refresh_rmw_on_load:
            return _refresh_cached_rmw(
                cache=cache,
                runname=runname,
                directories=directories,
                trackfiles=trackfiles,
                cache_file=cache_file,
            )
        return cache

    print(f"[{runname}] building rain radial cache")
    ds = Dataset(directories[runname])
    times_all = _get_times(ds)

    lat2d = np.asarray(ds.variables["XLAT"][0], dtype=float)
    lon2d = np.asarray(ds.variables["XLONG"][0], dtype=float)
    clat_all, clon_all = _read_track(trackfiles[runname])

    time_indices = _get_time_indices(times_all, timerange)
    times = times_all[time_indices]
    r_centers = 0.5 * (r_edges_km[:-1] + r_edges_km[1:])

    rain_hov = np.full((len(time_indices), len(r_centers)), np.nan)
    rmw_km = np.full(len(time_indices), np.nan)

    for n, it in enumerate(time_indices):
        print(f"[{runname}] rain {n+1}/{len(time_indices)} {times_all[it]}")
        clat = clat_all[it]
        clon = clon_all[it]
        radius_km, theta, radius_deg = _latlon_to_polar(lat2d, lon2d, clat, clon)

        rain_rate = _compute_rain_rate(ds, times_all, it)
        rain_hov[n, :] = _radial_mean(rain_rate, radius_km, r_edges_km)

        u10 = np.asarray(ds.variables["U10"][it], dtype=float)
        v10 = np.asarray(ds.variables["V10"][it], dtype=float)
        rmw_km[n] = _compute_rmw_from_tangential(
            u10=u10,
            v10=v10,
            theta=theta,
            radius_km=radius_km,
            r_edges_km=r_edges_km,
            rmw_search_km=rmw_search_km,
        )

    ds.close()

    out = {
        "runname": runname,
        "times": pd.DatetimeIndex(times),
        "r_edges_km": r_edges_km,
        "r_centers_km": r_centers,
        "rain_hov": rain_hov,
        "rmw_km": rmw_km,
        "rmw_search_km": rmw_search_km,
        "rmw_method": "tangential_wind",
    }
    save_pkl(out, cache_file)
    print(f"[{runname}] saved {cache_file}")
    return out


In [ ]:
def add_forward_integrated_budget_terms(cache, future_hours=6):
    times = pd.DatetimeIndex(cache["times"])
    tsec = np.asarray((times - times[0]).total_seconds(), dtype=float)

    def _forward_integral_2d(arr):
        arr = np.asarray(arr, dtype=float)
        out = np.full_like(arr, np.nan, dtype=float)
        for i, t0 in enumerate(times):
            t1 = t0 + pd.Timedelta(hours=future_hours)
            inds = np.where((times >= t0) & (times <= t1))[0]
            if len(inds) >= 2:
                out[i, :] = np.trapz(arr[inds, :], x=tsec[inds], axis=0)
        return out

    def _forward_change_2d(arr):
        arr = np.asarray(arr, dtype=float)
        out = np.full_like(arr, np.nan, dtype=float)
        for i, t0 in enumerate(times):
            target = t0 + pd.Timedelta(hours=future_hours)
            j = int(np.argmin(np.abs(times - target)))
            if times[j] > t0:
                out[i, :] = arr[j, :] - arr[i, :]
        return out

    for name in ["advh", "stretch", "vadv", "tilt", "resolved", "resid"]:
        cache[f"{name}_int{future_hours}h_s1"] = _forward_integral_2d(cache[f"{name}_hov_s2"])
        cache[f"{name}_int{future_hours}h_1e5"] = cache[f"{name}_int{future_hours}h_s1"] * 1e5

    cache[f"davor_{future_hours}h_s1"] = _forward_change_2d(cache["avor_hov_s1"])
    cache[f"davor_{future_hours}h_1e5"] = cache[f"davor_{future_hours}h_s1"] * 1e5

    return cache

In [ ]:

def build_layer_vortbudget_shared_caches(
    runname,
    directories,
    trackfiles,
    z1=1000.0,
    z2=5000.0,
    cache_dir="cache",
    r_edges_km=np.arange(0, 251, 5),
    timerange=None,
    rmw_search_km=(10, 150),
    core_radii_deg=None,
    overwrite=False,
    refresh_rmw_on_load=False,
):
    if core_radii_deg is None:
        core_radii_deg = {
            "inner1deg": (0.0, 1.0),
            "annulus1to2deg": (1.0, 2.0),
            "disk2deg": (0.0, 2.0),
        }

    ztag = f"{int(z1)}to{int(z2)}m"
    radial_cache_file = os.path.join(cache_dir, f"{runname}_vortbudget_{ztag}_rmw_radial.pkl")
    coremask_cache_file = os.path.join(cache_dir, f"{runname}_vortbudget_{ztag}_coremask.pkl")

    if os.path.exists(radial_cache_file) and not overwrite:
        print(f"[{runname}] loading existing radial vortbudget cache")
        radial = load_pkl(radial_cache_file)
        coremask = load_pkl(coremask_cache_file) if os.path.exists(coremask_cache_file) else None
        if refresh_rmw_on_load:
            radial = _refresh_cached_rmw(
                cache=radial,
                runname=runname,
                directories=directories,
                trackfiles=trackfiles,
                cache_file=radial_cache_file,
            )
        return radial, coremask

    print(f"[{runname}] building shared vortbudget caches for {z1:.0f}-{z2:.0f} m")

    ds = Dataset(directories[runname])
    times_all = _get_times(ds)

    lat2d = np.asarray(ds.variables["XLAT"][0], dtype=float)
    lon2d = np.asarray(ds.variables["XLONG"][0], dtype=float)
    clat_all, clon_all = _read_track(trackfiles[runname])
    cx_all, cy_all = _storm_motion_ms(times_all, clat_all, clon_all)

    time_indices = _get_time_indices(times_all, timerange)
    times = pd.DatetimeIndex(times_all[time_indices])
    r_centers = 0.5 * (r_edges_km[:-1] + r_edges_km[1:])
    nt = len(time_indices)
    nr = len(r_centers)

    avor_hov = np.full((nt, nr), np.nan)
    advh_hov = np.full((nt, nr), np.nan)
    stretch_hov = np.full((nt, nr), np.nan)
    vadv_hov = np.full((nt, nr), np.nan)
    tilt_hov = np.full((nt, nr), np.nan)
    resolved_hov = np.full((nt, nr), np.nan)
    rmw_km = np.full(nt, np.nan)

    coremask = {
        "runname": runname,
        "times": times,
        "z1": z1,
        "z2": z2,
        "core_radii_deg": core_radii_deg,
    }
    for region_name in core_radii_deg:
        coremask[region_name] = {
            "rain": np.full(nt, np.nan),
            "avor_s1": np.full(nt, np.nan),
            "advh_s2": np.full(nt, np.nan),
            "stretch_s2": np.full(nt, np.nan),
            "vadv_s2": np.full(nt, np.nan),
            "tilt_s2": np.full(nt, np.nan),
            "resolved_s2": np.full(nt, np.nan),
        }

    for n, it in enumerate(time_indices):
        print(f"[{runname}] shared vortbudget {n+1}/{nt} {times_all[it]}")

        clat = clat_all[it]
        clon = clon_all[it]
        radius_km, theta, radius_deg = _latlon_to_polar(lat2d, lon2d, clat, clon)

        rain_rate = _compute_rain_rate(ds, times_all, it)
        terms3d = _compute_budget_terms_3d(
            ds=ds,
            it=it,
            cx_ms=float(cx_all[it]),
            cy_ms=float(cy_all[it]),
        )

        rho_3d = terms3d["rho"]
        z_agl = terms3d["z_agl"]

        avor_layer = _layer_massweighted_mean(terms3d["avor"], rho_3d, z_agl, z1, z2)
        advh_layer = _layer_massweighted_mean(terms3d["advh"], rho_3d, z_agl, z1, z2)
        stretch_layer = _layer_massweighted_mean(terms3d["stretch"], rho_3d, z_agl, z1, z2)
        vadv_layer = _layer_massweighted_mean(terms3d["vadv"], rho_3d, z_agl, z1, z2)
        tilt_layer = _layer_massweighted_mean(terms3d["tilt"], rho_3d, z_agl, z1, z2)
        resolved_layer = _layer_massweighted_mean(terms3d["resolved"], rho_3d, z_agl, z1, z2)

        avor_hov[n, :] = _radial_mean(avor_layer, radius_km, r_edges_km)
        advh_hov[n, :] = _radial_mean(advh_layer, radius_km, r_edges_km)
        stretch_hov[n, :] = _radial_mean(stretch_layer, radius_km, r_edges_km)
        vadv_hov[n, :] = _radial_mean(vadv_layer, radius_km, r_edges_km)
        tilt_hov[n, :] = _radial_mean(tilt_layer, radius_km, r_edges_km)
        resolved_hov[n, :] = _radial_mean(resolved_layer, radius_km, r_edges_km)

        u10 = np.asarray(ds.variables["U10"][it], dtype=float)
        v10 = np.asarray(ds.variables["V10"][it], dtype=float)
        rmw_km[n] = _compute_rmw_from_tangential(
            u10=u10,
            v10=v10,
            theta=theta,
            radius_km=radius_km,
            r_edges_km=r_edges_km,
            rmw_search_km=rmw_search_km,
        )

        for region_name, (rmin_deg, rmax_deg) in core_radii_deg.items():
            coremask[region_name]["rain"][n] = _apply_radius_mask(rain_rate, radius_deg, rmin_deg, rmax_deg)
            coremask[region_name]["avor_s1"][n] = _apply_radius_mask(avor_layer, radius_deg, rmin_deg, rmax_deg)
            coremask[region_name]["advh_s2"][n] = _apply_radius_mask(advh_layer, radius_deg, rmin_deg, rmax_deg)
            coremask[region_name]["stretch_s2"][n] = _apply_radius_mask(stretch_layer, radius_deg, rmin_deg, rmax_deg)
            coremask[region_name]["vadv_s2"][n] = _apply_radius_mask(vadv_layer, radius_deg, rmin_deg, rmax_deg)
            coremask[region_name]["tilt_s2"][n] = _apply_radius_mask(tilt_layer, radius_deg, rmin_deg, rmax_deg)
            coremask[region_name]["resolved_s2"][n] = _apply_radius_mask(resolved_layer, radius_deg, rmin_deg, rmax_deg)

    ds.close()

    tsec = np.asarray((times - times[0]).total_seconds(), dtype=float)
    davor_dt_hov = np.full_like(avor_hov, np.nan)
    for j in range(nr):
        y = avor_hov[:, j]
        good = np.isfinite(y)
        if np.count_nonzero(good) >= 3:
            davor_dt_hov[good, j] = np.gradient(y[good], tsec[good], edge_order=2)

    resid_hov = davor_dt_hov - resolved_hov

    radial = {
        "runname": runname,
        "times": times,
        "r_edges_km": r_edges_km,
        "r_centers_km": r_centers,
        "rmw_km": rmw_km,
        "rmw_search_km": rmw_search_km,
        "rmw_method": "tangential_wind",
        "z1": z1,
        "z2": z2,
        "avor_hov_s1": avor_hov,
        "avor_hov_1e5": avor_hov * 1e5,
        "advh_hov_s2": advh_hov,
        "stretch_hov_s2": stretch_hov,
        "vadv_hov_s2": vadv_hov,
        "tilt_hov_s2": tilt_hov,
        "resolved_hov_s2": resolved_hov,
        "davor_dt_hov_s2": davor_dt_hov,
        "resid_hov_s2": resid_hov,
        "advh_hov_1e10": advh_hov * 1e10,
        "stretch_hov_1e10": stretch_hov * 1e10,
        "vadv_hov_1e10": vadv_hov * 1e10,
        "tilt_hov_1e10": tilt_hov * 1e10,
        "resolved_hov_1e10": resolved_hov * 1e10,
        "davor_dt_hov_1e10": davor_dt_hov * 1e10,
        "resid_hov_1e10": resid_hov * 1e10,
    }

    for region_name in coremask["core_radii_deg"]:
        reg = coremask[region_name]
        arr = reg["avor_s1"]
        good = np.isfinite(arr)
        reg["davor_dt_s2"] = np.full_like(arr, np.nan, dtype=float)
        if np.count_nonzero(good) >= 3:
            reg["davor_dt_s2"][good] = np.gradient(arr[good], tsec[good], edge_order=2)
        reg["resid_s2"] = reg["davor_dt_s2"] - reg["resolved_s2"]

    save_pkl(radial, radial_cache_file)
    save_pkl(coremask, coremask_cache_file)
    print(f"[{runname}] saved {radial_cache_file}")
    print(f"[{runname}] saved {coremask_cache_file}")

    return radial, coremask


In [ ]:
def build_peak_profiles_rain_budget6h(
    peak_specs,
    rain_radial,
    vortbudget_radial,
    future_hours=6,
    rain_window_hours=3,
    rain_mode="mean",
    level_tag="1to5km",
    base_peak_profiles=None,
):
    """
    Build peak-centered radial profiles with layer-specific budget keys.

    The rainfall profile is identical for each layer because it comes from the
    rain/RMW radial cache. Budget variables are written with the requested
    level tag, for example:

    - stretch6_1to5km
    - stretch6_0to1km
    - davor6_1to5km
    - davor6_0to1km

    Passing base_peak_profiles allows a second layer to be added to an
    existing profile dictionary without overwriting the first layer.
    """
    peak_profiles = copy.deepcopy(base_peak_profiles) if base_peak_profiles is not None else {}
    hour_tag = f"{future_hours}"

    for peak_key, spec in peak_specs.items():
        runname = spec["runname"]
        tsel = pd.Timestamp(spec["time"])

        rain_dat = rain_radial[runname]
        rain_times = pd.DatetimeIndex(rain_dat["times"])
        it0_rain = int(np.argmin(np.abs(rain_times - tsel)))
        t1_rain = tsel + pd.Timedelta(hours=rain_window_hours)

        rain_inds = np.where((rain_times >= tsel) & (rain_times <= t1_rain))[0]
        if len(rain_inds) == 0:
            raise ValueError(f"No rain times found in [{tsel}, {t1_rain}] for {runname}")

        r_common = np.asarray(rain_dat["r_centers_km"], dtype=float)
        rain_block = np.asarray(rain_dat["rain_hov"][rain_inds, :], dtype=float)

        if rain_mode == "mean":
            rain_prof = np.nanmean(rain_block, axis=0)
        elif rain_mode == "sum":
            rain_prof = np.nansum(rain_block, axis=0)
        else:
            raise ValueError("rain_mode must be 'mean' or 'sum'")

        rmw_now = float(rain_dat["rmw_km"][it0_rain])

        bud = vortbudget_radial[runname]
        bud_times = pd.DatetimeIndex(bud["times"])
        it0_bud = int(np.argmin(np.abs(bud_times - tsel)))
        r_bud = np.asarray(bud["r_centers_km"], dtype=float)

        if peak_key not in peak_profiles:
            peak_profiles[peak_key] = {
                "title": spec["title"],
                "runname": runname,
                "valid_time": tsel,
                "radius_km": r_common,
                "rmw_km": rmw_now,
                "rmw15_km": 1.5 * rmw_now,
                "rain": rain_prof,
            }
        else:
            # Keep shared metadata/rainfall synchronized when adding another layer.
            peak_profiles[peak_key].update({
                "title": spec["title"],
                "runname": runname,
                "valid_time": tsel,
                "radius_km": r_common,
                "rmw_km": rmw_now,
                "rmw15_km": 1.5 * rmw_now,
                "rain": rain_prof,
            })

        peak_profiles[peak_key].update({
            f"avor_{level_tag}": _interp_to_r(r_bud, bud["avor_hov_1e5"][it0_bud, :], r_common),
            f"davor{hour_tag}_{level_tag}": _interp_to_r(r_bud, bud[f"davor_{future_hours}h_1e5"][it0_bud, :], r_common),
            f"stretch{hour_tag}_{level_tag}": _interp_to_r(r_bud, bud[f"stretch_int{future_hours}h_1e5"][it0_bud, :], r_common),
            f"advh{hour_tag}_{level_tag}": _interp_to_r(r_bud, bud[f"advh_int{future_hours}h_1e5"][it0_bud, :], r_common),
            f"vadv{hour_tag}_{level_tag}": _interp_to_r(r_bud, bud[f"vadv_int{future_hours}h_1e5"][it0_bud, :], r_common),
            f"tilt{hour_tag}_{level_tag}": _interp_to_r(r_bud, bud[f"tilt_int{future_hours}h_1e5"][it0_bud, :], r_common),
            f"resid{hour_tag}_{level_tag}": _interp_to_r(r_bud, bud[f"resid_int{future_hours}h_1e5"][it0_bud, :], r_common),
        })

    return peak_profiles


In [ ]:
def _smooth_nan_1d(y, k):
    y = np.asarray(y, dtype=float)
    if k <= 1:
        return y.copy()
    kernel = np.ones(k, dtype=float) / float(k)
    good = np.isfinite(y).astype(float)
    y0 = np.where(np.isfinite(y), y, 0.0)
    num = np.convolve(y0, kernel, mode="same")
    den = np.convolve(good, kernel, mode="same")
    out = np.full_like(y, np.nan, dtype=float)
    m = den > 0
    out[m] = num[m] / den[m]
    return out


def _prepare_peak_budget_series(dat, level_tag="0to1km", smooth_bins=1, future_hours=6):
    r = np.asarray(dat["radius_km"], dtype=float)
    rain = _smooth_nan_1d(dat["rain"], smooth_bins)

    adv_flux = _smooth_nan_1d(
        np.asarray(dat[f"advh{future_hours}_{level_tag}"], dtype=float)
        + np.asarray(dat[f"stretch{future_hours}_{level_tag}"], dtype=float),
        smooth_bins,
    )
    nonadv_flux = _smooth_nan_1d(
        np.asarray(dat[f"vadv{future_hours}_{level_tag}"], dtype=float)
        + np.asarray(dat[f"tilt{future_hours}_{level_tag}"], dtype=float),
        smooth_bins,
    )
    davor = _smooth_nan_1d(
        np.asarray(dat[f"davor{future_hours}_{level_tag}"], dtype=float),
        smooth_bins,
    )

    return {
        "radius_km": r,
        "rain": rain,
        "adv_flux": adv_flux,
        "nonadv_flux": nonadv_flux,
        "davor": davor,
        "rmw_km": float(dat["rmw_km"]) if np.isfinite(dat["rmw_km"]) else np.nan,
        "title": dat["title"],
    }


def _radial_area_weighted_mean(r, y, r0, r1):
    """Area-weighted mean of a radial profile y(r) over [r0, r1]."""
    r = np.asarray(r, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(r) & np.isfinite(y) & (r >= r0) & (r <= r1)
    if np.sum(m) < 2:
        return np.nan

    rr = r[m]
    yy = y[m]
    order = np.argsort(rr)
    rr = rr[order]
    yy = yy[order]

    w = rr
    num = np.trapz(yy * w, x=rr)
    den = np.trapz(w, x=rr)
    if not np.isfinite(den) or den == 0:
        return np.nan
    return num / den


def plot_peak_region_summary_vortflux_budget6h_ams(
    peak_profiles,
    peak_order,
    level_tag="0to1km",
    future_hours=6,
    figsize=(11.8, 8.2),
    smooth_bins=3,
    rmin_stat_km=15.0,
    rmax_stat_km=200.0,
    display_names=None,
    savepath=None,
):
    """
    Manuscript RMW-relative budget summary.

    Inside region: 15 km to RMW.
    Outside region: RMW to rmax_stat_km, set to 200 km for the manuscript.
    """
    if display_names is None:
        display_names = {
            "control": "Control",
            "diurnal": "Diurnal",
            "dimp_first": "D_imp1 first",
            "dimp_second": "D_imp1 second",
        }

    stats = {
        "rain_in": [], "rain_out": [],
        "adv_in": [], "adv_out": [],
        "nonadv_in": [], "nonadv_out": [],
        "davor_in": [], "davor_out": [],
    }
    labels = []

    for peak_key in peak_order:
        d = _prepare_peak_budget_series(
            peak_profiles[peak_key],
            level_tag=level_tag,
            smooth_bins=smooth_bins,
            future_hours=future_hours,
        )
        r = d["radius_km"]
        rmw = d["rmw_km"]
        labels.append(display_names.get(peak_key, peak_key))

        if (not np.isfinite(rmw)) or (rmw <= rmin_stat_km) or (rmax_stat_km <= rmw):
            for key in stats:
                stats[key].append(np.nan)
            continue

        # Inside RMW: 15 km to RMW.
        stats["rain_in"].append(_radial_area_weighted_mean(r, d["rain"], rmin_stat_km, rmw))
        stats["adv_in"].append(_radial_area_weighted_mean(r, d["adv_flux"], rmin_stat_km, rmw))
        stats["nonadv_in"].append(_radial_area_weighted_mean(r, d["nonadv_flux"], rmin_stat_km, rmw))
        stats["davor_in"].append(_radial_area_weighted_mean(r, d["davor"], rmin_stat_km, rmw))

        # Outside RMW: RMW to 200 km.
        stats["rain_out"].append(_radial_area_weighted_mean(r, d["rain"], rmw, rmax_stat_km))
        stats["adv_out"].append(_radial_area_weighted_mean(r, d["adv_flux"], rmw, rmax_stat_km))
        stats["nonadv_out"].append(_radial_area_weighted_mean(r, d["nonadv_flux"], rmw, rmax_stat_km))
        stats["davor_out"].append(_radial_area_weighted_mean(r, d["davor"], rmw, rmax_stat_km))

    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True)
    axes = axes.ravel()
    x = np.arange(len(labels))
    width = 0.36

    panel_cfg = [
        ("rain_in", "rain_out", r"Rain rate (mm h$^{-1}$)", "Rain rate", "(a)"),
        ("adv_in", "adv_out", r"6-h contribution ($10^{-5}$ s$^{-1}$)", "Advective flux convergence", "(b)"),
        ("nonadv_in", "nonadv_out", r"6-h contribution ($10^{-5}$ s$^{-1}$)", "Nonadvective flux convergence", "(c)"),
        ("davor_in", "davor_out", r"6-h contribution ($10^{-5}$ s$^{-1}$)", "Absolute vorticity change", "(d)"),
    ]

    for ax, (kin, kout, ylabel, title, panel_label) in zip(axes, panel_cfg):
        yin = np.asarray(stats[kin], dtype=float)
        yout = np.asarray(stats[kout], dtype=float)

        ax.bar(x - width/2, yin, width=width, color="0.35", edgecolor="0.15", linewidth=0.8, label="Inside RMW")
        ax.bar(x + width/2, yout, width=width, color="white", edgecolor="0.15", linewidth=0.8, hatch="//", label="RMW–200 km")

        if "rain" not in kin:
            ax.axhline(0.0, color="0.35", lw=0.9)
        ax.grid(True, axis="y", alpha=0.22, linewidth=0.55)
        ax.set_ylabel(ylabel, fontsize=10.5)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.text(0.98, 0.96, panel_label, transform=ax.transAxes, ha="right", va="top", fontsize=12, fontweight="bold")

    for ax in axes[2:]:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=10)
    for ax in axes[:2]:
        ax.set_xticks(x)
        ax.tick_params(labelbottom=False)

    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor="0.35", edgecolor="0.15", linewidth=0.8, label="Inside RMW"),
        plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor="0.15", hatch="//", linewidth=0.8, label="RMW–200 km"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.995), fontsize=10.5)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if savepath is not None:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.show()
    return fig, axes, stats


## Build/load derived caches

In [ ]:
# -----------------------------------------------------------------------------
# Build or load the derived caches needed for the manuscript RMW-budget figure
# -----------------------------------------------------------------------------
rain_radial = {}
vortbudget_radial = {}

for runname in runnamelistuse:
    print(f"[{runname}] loading/building rainfall/RMW cache")
    rain_radial[runname] = build_rain_rmw_radial_cache(
        runname=runname,
        directories=directoriesuse,
        trackfiles=trackfilesuse,
        cache_dir=cache_dir,
        r_edges_km=r_edges_km,
        timerange=timerange,
        rmw_search_km=rmw_search_km,
        overwrite=OVERWRITE_CACHE,
        refresh_rmw_on_load=REFRESH_RMW_ON_LOAD,
    )

    print(f"[{runname}] loading/building 0-1 km vorticity-budget cache")
    radial_cache, _ = build_layer_vortbudget_shared_caches(
        runname=runname,
        directories=directoriesuse,
        trackfiles=trackfilesuse,
        z1=z1_budget,
        z2=z2_budget,
        cache_dir=cache_dir,
        r_edges_km=r_edges_km,
        timerange=timerange,
        rmw_search_km=rmw_search_km,
        overwrite=OVERWRITE_CACHE,
        refresh_rmw_on_load=REFRESH_RMW_ON_LOAD,
    )
    vortbudget_radial[runname] = add_forward_integrated_budget_terms(
        radial_cache,
        future_hours=future_hours,
    )


## Build peak profiles and plot the manuscript figure

In [ ]:
# Build peak-centered radial profiles used by the summary figure.
peak_profiles_budget6h = build_peak_profiles_rain_budget6h(
    peak_specs=peak_specs,
    rain_radial=rain_radial,
    vortbudget_radial=vortbudget_radial,
    future_hours=future_hours,
    rain_window_hours=rain_window_hours,
    rain_mode="mean",
    level_tag="0to1km",
)


In [ ]:
# Reproduce the manuscript RMW-relative 0-1 km vorticity-budget summary figure.
fig13_path = os.path.join(figure_dir, "fig13_vortbudget_rmw_summary.png")
fig, axes, stats = plot_peak_region_summary_vortflux_budget6h_ams(
    peak_profiles=peak_profiles_budget6h,
    peak_order=["control", "diurnal", "dimp_first", "dimp_second"],
    level_tag="0to1km",
    future_hours=future_hours,
    figsize=(11.8, 8.2),
    smooth_bins=3,
    rmin_stat_km=rmin_stat_km,
    rmax_stat_km=rmax_stat_km,
    savepath=fig13_path,
)
print(f"Saved {fig13_path}")
